## Praktikum 1
##### Landmarkenerkennung für Gesichter

In diesem Praktikum betrachten wir die Grundlagen der Gesichtserkennung.

[C++-Implementierung zur Gesichtserkennung mit 3.000 FPS](https://github.com/luoyetx/face-alignment-at-3000fps)

[Eine schnellere Gesichtserkennung (nicht für dieses Praktikum empfohlen)](http://dlib.net/faq.html#Whyisdlibslow)

<div style="padding: 10px; border-left: 5px solid #B777FA; background-color: #5C18FA; border-radius: 4px">
  🐍 Bitte nutzen Sie die virtuelle Umgebung: <b>DLACV_2526</b>
</div>

In [12]:
# imports
import sys
import math
import dlib
import cv2
import matplotlib
import numpy as np
from utils import faceBlendCommon as fbc
from utils.renderFace import renderFace
from utils.dataPath import DATA_PATH
from utils.dataPath import MODEL_PATH
import matplotlib.pyplot as plt
%matplotlib inline

#### Teil 3: Gesichter in Videos
Bisher haben wir die Gesichts- und Landmarkendetektoren nur auf Bilder angewandt - aber wenn Sie an Realweltbeispiele denken, machen diese sich fast immer einen Videostream zunutze. Und auch wenn DLIB mit einigen kleinen Anpassungen mit etwa 1.000 FPS laufen kann, geht es schwer auf die Hardware, jeden einzelnen Videoframe zu analysieren - somit ist weniger DLIB als nicht eher die Hardware der Bottleneck. Um dem entgegenzuwirken, kann man die Auflösung oder den Intervall, in welchem wir tatsächlich auf Gesichter überprüfen, anpassen. Zusätzlich kann DLIB mit einigen Flags auf bestimmter Hardware nochmal etwas schneller laufen (siehe [Eine schnellere Gesichtserkennung (nicht für dieses Praktikum empfohlen)](http://dlib.net/faq.html#Whyisdlibslow)).

Da die Landmarken dazu tendieren, instabil auszusehen (also hin- und herzuwackeln), lassen sich die Landmarken in Videos beliebig weiter stabilisieren, unter anderem mit [Kalman-Filterung](https://www.ultralytics.com/glossary/kalman-filter-kf), [Optical Flow](https://www.ultralytics.com/de/glossary/optical-flow) oder [Image Pyramids](https://docs.opencv.org/4.x/dc/dff/tutorial_py_pyramids.html). Dabei macht man sich dann die Distanz zwischen den Augen als Anker zunutze. Darauf wird in diesem Praktikum allerdings nicht weiter eingegangen.

<div style="
    padding: 12px; 
    border-left: 5px solid #e8f4fc; 
    background-color: #3498; 
    border-radius: 4px;
">
  📌 <b>Aufgabe 3.1:</b><br>
  Passen Sie den Code in der <code>while</code>-Schleife an:<br>
  • Sodass nur alle <code>SKIP_FRAME</code> Frames der Gesichtsdetektor aufgerufen wird.<br>
  • Sodass das eingelesene Bild um mit der <code>RESIZE_SCALE</code> geschrumpft wird und beim Erstellen des Rechtecks wieder vergrößert wird. Nutzen Sie für die Interpolation <code>cv2.INTER_LINEAR</code>.
</div>

In [13]:
# Pfad zum Landmarkendetektor
PREDICTOR_PATH = "data/models/shape_predictor_68_face_landmarks.dat"

CAP_INDEX = 0  # Verändern Sie diesen Wert, um eine andere Webcam auszuwählen. Alternativ können Sie auch einen Pfad zu einer Videodatei angeben.

# Parameter fürs Auflösung reduzieren und Überspringen von Frames
RESIZE_HEIGHT = 240
SKIP_FRAMES = 1

In [14]:
# Fenster initialisieren
winName = "Fast Facial Landmark Detector"

# Video-Capture initialisieren
cap = cv2.VideoCapture(CAP_INDEX)

# Überprüfe Kamera
if (cap.isOpened() is False):
    print("Kann keine Verbindung zur Kamera herstellen.")
    sys.exit()

# Startwert für die FPS-Anzeige. Wird nach 100 Frames neu berechnet.
fps = 30.0

# Lese ersten Frame aus
ret, im = cap.read()

# Feste Höhe für den Input unseres Gesichtsdetektors
if ret == True:
    height = im.shape[0]
    # Berechne Skalierung anhand der Höhe
    RESIZE_SCALE = float(height)/RESIZE_HEIGHT
    size = im.shape[0:2]
else:
    print("Kann Frame nicht auslesen.")
    sys.exit()

# Detektoren initialisieren
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(PREDICTOR_PATH)

# Tick-Counter initialisieren
t = cv2.getTickCount()
count = 0

# try/finally stellt sicher, dass die Kamera auch bei einem Kernel-Interrupt
# (z.B. "Interrupt Kernel" statt Esc im Kamerafenster) wieder freigegeben wird.
# Ohne das bleibt die Kamera für den Rest der Kernel-Sitzung blockiert.
try:
    # Lese Frames aus, bis der Nutzer abbricht
    while(True):
        if count==0:
          t = cv2.getTickCount()

        # Lese Frame aus
        ret, im = cap.read()
        imDlib = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

        # Schrumpfe das Originalbild (cv2.resize erwartet dsize als (width, height))
        imSmall = cv2.resize(imDlib, np.array((int(imDlib.shape[1]/RESIZE_SCALE),int(imDlib.shape[0]/RESIZE_SCALE))) , interpolation=cv2.INTER_LINEAR) 
        imSmallDlib = cv2.cvtColor(imSmall, cv2.COLOR_BGR2RGB)

        # Nutze den Detektor nur alle n Frames. Je höher n, desto weniger Berechnungen werden durchgeführt.
        faces = detector(imSmallDlib,0)
        if count%SKIP_FRAMES == 0:
          # Überprüfe alle Gesichter
          for face in faces:
            # Wir skalieren das Rechteck wieder hoch (nicht das Bild - face ist kein Bild, sondern ein dlib.rectangle)
            newRect = dlib.rectangle(int(face.left()*RESIZE_SCALE),
                                      int(face.top()*RESIZE_SCALE),
                                      int(face.right()*RESIZE_SCALE),
                                      int(face.bottom()*RESIZE_SCALE))

            # Berechne Landmarken
            shape = predictor(imDlib, newRect)
            # Zeichne die Polylinien
            renderFace(im, shape)

        # Schreibe aktuelle FPS auf das Bild
        cv2.putText(im, "{0:.2f}-fps".format(fps), (50, size[0]-50), cv2.FONT_HERSHEY_COMPLEX, 1.5, (0, 0, 255), 3)
        # Zeige das Bild an
        cv2.imshow(winName, im)

        # Überprüfe auf Taste
        key = cv2.waitKey(1) & 0xFF

        # Stoppe das Programm
        if key==27 or cv2.getWindowProperty(winName, cv2.WND_PROP_VISIBLE) <1:  # ESC oder 'x'
          break

        # Erhöhe Frame-Zähler
        count = count + 1
        # Berechne alle 100 Frames die FPS neu
        if (count == 100):
          t = (cv2.getTickCount() - t)/cv2.getTickFrequency()
          fps = 100.0/t
          count = 0
finally:
    cv2.destroyAllWindows()
    cap.release()

<div style="
    padding: 12px; 
    border-left: 5px solid #e8f4fc; 
    background-color: #3498; 
    border-radius: 4px;
">
  📌 <b>Aufgabe 3.2:</b><br>
  • Für ein späteres Praktikum benötigen wir einen Datensatz an Gesichtern für die Gesichtserkennung.<br>
  Bitte gehen Sie im Ilias-Kurs auf <i>Praktikum > Abgabe - Bilder für Gesichtserkennung</i> und folgen Sie den Anweisungen dort.
</div>